# Inverse-Volatility Risk Parity Backtest: ES, Gold, 10Y Notes

Each asset is weighted by `1 / rolling volatility`, which is "naive" risk parity because it ignores correlations. The portfolio is compared with an equal-weight benchmark. **Downloads, weights, the backtest, statistics, charts and the Plotly theme all come from `func_lib.py`.**

| func_lib helper | Purpose |
|---|---|
| `download_close` | Adjusted closes from yfinance, one column per ticker |
| `inverse_vol_weights` | 1/vol weights; optional weekly or monthly rebalance |
| `backtest_weights` | Weights lagged one day (no look-ahead), turnover and transaction costs |
| `performance_stats`, `stats_table_figure` | CAGR, volatility, Sharpe/Sortino on excess returns, max drawdown, Calmar; dark table |
| `weights_area_figure`, `equity_drawdown_figure` | Stacked weights; equity and underwater charts |

**Bug fixes vs. the original**
- **Look-ahead bias:** returns were multiplied by *same-day* weights, which already contain that day's return. Weights are now lagged one day.
- **Cell order:** annualized return and volatility were computed before `portfolio_returns` existed.
- **Risk-free rate:** `^IRX` came back as a DataFrame, so every ratio became a Series. It is now a daily rate series.
- **Consistent ratios:** Sharpe and Sortino both use excess returns. Downside deviation is the proper semi-deviation. Calmar uses CAGR.
- The plot title said "2023" for a 2015–2025 backtest.

*Note:* `=F` tickers are continuous front-month futures, so roll dates add small price jumps. Returns are fully collateralized futures returns.

In [ ]:
"""
Imports, plus func_lib for downloads, risk parity weights, the backtest and the charts.
"""
import os
import sys

import numpy as np
import pandas as pd

# Make func_lib importable from this folder or its parent, without duplicate sys.path entries
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)

import func_lib   # importing it also applies the dark Plotly theme ('quant_dark')
from func_lib import TRADING_DAYS

In [ ]:
"""
Settings for the whole notebook: the three futures, date range, volatility window, how often to
rebalance and the trading cost.
"""
# ---- Configuration ----
TICKERS = ['ES=F', 'GC=F', 'ZN=F']   # S&P 500 e-mini, gold, 10-year T-note futures
START_DATE, END_DATE = '2015-01-01', '2025-12-31'
VOL_WINDOW = 36        # trading days in the rolling volatility estimate
REBALANCE = None       # None = daily, 'W' = weekly, 'M' = monthly (last trading day of period)
COST_BPS = 0.0         # transaction cost per unit of one-way turnover, in basis points

## 1. Data

In [ ]:
"""
Downloads futures prices for the S&P 500 e-mini, gold and 10-year T-notes, turns them into daily
returns, and converts the 13-week T-bill yield into a daily risk-free rate.
"""
# Forward-fill across exchange-holiday mismatches, then drop rows before every series starts
prices = func_lib.download_close(TICKERS, START_DATE, END_DATE)[TICKERS].ffill().dropna()
returns = prices.pct_change().dropna()

# ^IRX is the 13-week T-bill yield in annual %; convert it to a daily rate on the return dates
irx = func_lib.download_close('^IRX', START_DATE, END_DATE).squeeze('columns')
rf_daily = (irx / 100 / TRADING_DAYS).reindex(returns.index).ffill().bfill()

print(f'{len(returns):,} daily returns, {returns.index[0]:%Y-%m-%d} to {returns.index[-1]:%Y-%m-%d}')
print(f'Average risk-free rate: {irx.mean() / 100:.2%} p.a.')

## 2. Inverse-volatility weights and backtest

In [ ]:
"""
Risk parity weights: each asset is weighted by 1 / its recent volatility, so calmer assets get
more weight. Weights use only yesterday's data (no look-ahead). Builds the equal-weight benchmark
and charts the weights over time.
"""
target_weights = func_lib.inverse_vol_weights(returns, VOL_WINDOW, REBALANCE)
rp_returns, weights, turnover = func_lib.backtest_weights(target_weights, returns, COST_BPS)
ew_returns = returns.loc[weights.index].mean(axis=1)   # equal-weight benchmark (daily rebalanced)

assert np.allclose(weights.sum(axis=1), 1.0)
print(f'Average daily one-way turnover: {turnover.mean():.4f}')
func_lib.weights_area_figure(weights, 'Risk Parity Weights (inverse volatility)').show()

## 3. Performance

In [ ]:
"""
Performance table for risk parity and equal weight: return, volatility, Sharpe, Sortino and drawdown.
"""
stats = pd.DataFrame({
    'Risk Parity': func_lib.performance_stats(rp_returns, rf_daily),
    'Equal Weight': func_lib.performance_stats(ew_returns, rf_daily),
})
func_lib.stats_table_figure(
    stats, 'Performance Summary',
    pct_rows=('Total Return', 'CAGR', 'Annualized Return (arith.)', 'Annualized Volatility',
              'Downside Deviation', 'Max Drawdown', 'Positive Days'),
).show()

In [ ]:
"""
Growth of $1 and drawdowns for both portfolios over the full period.
"""
period = f'{rp_returns.index[0]:%Y}-{rp_returns.index[-1]:%Y}'
func_lib.equity_drawdown_figure({'Risk Parity': rp_returns, 'Equal Weight': ew_returns},
                                f'Cumulative Returns and Drawdowns ({period})').show()